In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
import ce # type: ignore
import linear # type: ignore
import softmax # type: ignore
from common import repeat, test_checkgrad_2, test_compare_2, test_model_sgd_2 # type: ignore
from approx import approx # type: ignore

In [ ]:
class Per_Lin_Softmax_CE_Autograd(nn.Module):
    """The version relying fully on PyTorch to handle `forward` and `backward` passes."""
    
    def __init__(self, in_features, out_features):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.loss_ = nn.CrossEntropyLoss(reduction='mean')

    def forward(self, x):
        return self.model(x)
    
    def model(self, x):
        return self.linear(x)

    def loss(self, p, t):
        return self.loss_(p, t)

    def predict(self, x):
        with tr.no_grad():
            return self.model(x).argmax(dim=1)

In [ ]:

class Per_Lin_Softmax_CE_Backward(nn.Module):
    """
    The version where the `forward` and `backward` passes for each operation are written manually, 
    but PyTorch's autograd still orchestrates the overall gradient flow through the computational graph.
    """

    def __init__(self, in_features, out_features, W: tr.Tensor = None, b: tr.Tensor = None) -> None:
        super().__init__()

        self.linear = linear.Linear(in_features, out_features, W, b)
        self.softmax = softmax.Softmax()
        self.loss_ = ce.CE(reduction='mean')

    def forward(self, x):
        return self.model(x)
    
    def model(self, x):
        return self.linear(x)
    
    def loss(self, p, t):
        return self.loss_(self.softmax(p), t)

    def predict(self, x):
        with tr.no_grad():
            return self.model(x).argmax(dim=1)       

$$ \text{model function: linear} $$

$$ \mathbb{R} \times \mathbb{R} \to \mathbb{R}, \quad z(w, b) = xw + b $$

$$ 
\mathbb{R}^m \times \mathbb{R}^n \to \mathbb{R}^n, \quad z(\mathbf{w}, \mathbf{b}) = 
X\mathbf{w} + \mathbf{b}
$$

$$ \frac{\partial z}{\partial \mathbf{w}} = X $$

$$ \frac{\partial z}{\partial \mathbf{b}} = \mathbf{I}_{n} $$

$$ 
dz = 
\frac{\partial z}{\partial \mathbf{w}} d\mathbf{w} + \frac{\partial z}{\partial \mathbf{b}} d\mathbf{b} 
$$

$$ \text{activation function: softmax} $$

$$ \mathbb{R}^n \to \mathbb{R}, \quad p_{i}(\mathbf{z}) = \frac{e^{z_i}}{\sum_{k} e^{z_k}} $$

$$ \mathbb{R}^n \to \mathbb{R}^n, \quad p(\mathbf{z}) = (p_1(\mathbf{z}), p_2(\mathbf{z}), \ldots, p_n(\mathbf{z})) $$

$$
\frac{\partial p_i}{\partial z_j} = p_i (\delta_{ij} - p_j) =
\begin{dcases}
    i = j & \frac{\partial p_i}{\partial z_i} = 
    \frac{e^{z_i} \sum_{k} e^{z_k} - e^{z_i} e^{z_i}}{\sum_{k} e^{z_k} \sum_{k} e^{z_k}} =
    p_i (1 - p_i) \\
    \\
    i \neq j & \frac{\partial p_i}{\partial z_j} = 
    \frac{e^{z_j}}{\sum_{k} e^{z_k} \sum_{k} e^{z_k}} = 
    p_i (0 - p_j)\\
\end{dcases}
$$

$$
\frac{d\mathbf{p}}{d\mathbf{z}} = J_p(\mathbf{z}) =
\text{diag}(\mathbf{p}) - \mathbf{p} \mathbf{p}^\top =
\begin{pmatrix}
    p_1 (1 - p_1) & -p_1 p_2 & \cdots & -p_1 p_n \\
    -p_2 p_1 & p_2 (1 - p_2) & \cdots & -p_2 p_n \\
    \vdots & \vdots & \ddots & \vdots \\
    -p_n p_1 & -p_n p_2 & \cdots & p_n (1 - p_n) \\
\end{pmatrix}
$$

$$ d\mathbf{p} = J_p(\mathbf{z}) \cdot d\mathbf{z} $$

$$ \text{loss funcion: cross-entropy} $$
$$ \mathbf{t_H} = \text{one-hot encoded vector} $$

$$ \mathbf{t}_{1} = (1, 0, \dots, 0) $$
$$ \mathbf{t}_{2} = (0, 1, \dots, 0) $$
$$ \cdots $$
$$ \mathbf{t}_{n} = (0, 0, \dots, 1) $$

$$
(0, 1)^n \times \{0, 1\}^n \to \mathbb{R}, \quad L(\mathbf{p}, \mathbf{t}_H) = 
-\sum_i t_i \, \ln(p_i) =
-\left \langle \mathbf{t}_H, \ln(\mathbf{p}) \right \rangle
$$

$$ 
\frac{\partial L}{\partial \mathbf{p}} = 
-\mathbf{t}_H \oslash \mathbf{p}
$$


$$
\frac{\partial L}{\partial \mathbf{z}} = 
\frac{\partial L}{\partial \mathbf{p}} \frac{\partial \mathbf{p}}{\partial \mathbf{z}} =
-\Big( \mathbf{t}_H \oslash \mathbf{p} \Big) \cdot \Big( \text{diag}(\mathbf{p}) - \mathbf{p} \mathbf{p}^\top \Big)^\top =
\mathbf{p} - \mathbf{t}_H
$$

$$ \text{Frobenius equality} $$

$$ \left \langle \frac{\partial F}{\partial \mathbf{w}}, d\mathbf{w} \right \rangle + 
\left \langle \frac{\partial F}{\partial \mathbf{b}}, d\mathbf{b} \right \rangle = 
\left \langle \frac{\partial F}{\partial L}, dL \right \rangle $$

$$ \\[2em] $$

$$ \left \langle \frac{\partial F}{\partial L}, dL \right \rangle = \frac{\partial F}{\partial L} \left \langle \frac{\partial L}{\partial \mathbf{z}}, d\mathbf{z} \right \rangle =  $$

$$ 
\frac{\partial F}{\partial L} \left \langle \frac{\partial L}{\partial \mathbf{z}}, \frac{\partial \mathbf{z}}{\partial \mathbf{w}} d\mathbf{w} + \frac{\partial \mathbf{z}}{\partial \mathbf{b}} d\mathbf{b} \right \rangle = 
$$

$$ 
\frac{\partial F}{\partial L} \left \langle \frac{\partial L}{\partial \mathbf{z}}, \frac{\partial \mathbf{z}}{\partial \mathbf{w}} d\mathbf{w} \right \rangle + 
\frac{\partial F}{\partial L} \left \langle \frac{\partial L}{\partial \mathbf{z}},\frac{\partial \mathbf{z}}{\partial \mathbf{b}} d\mathbf{b} \right \rangle = 
$$

$$ 
\frac{\partial F}{\partial L} \left \langle \Bigg( \frac{\partial \mathbf{z}}{\partial \mathbf{w}} \Bigg)^\top \frac{\partial L}{\partial \mathbf{z}}, d\mathbf{w} \right \rangle + 
\frac{\partial F}{\partial L} \left \langle \Bigg( \frac{\partial \mathbf{z}}{\partial \mathbf{b}} \Bigg)^\top \frac{\partial L}{\partial \mathbf{z}}, d\mathbf{b} \right \rangle = 
$$

$$ 
\left \langle \frac{\partial F}{\partial L} \Bigg( \frac{\partial \mathbf{z}}{\partial \mathbf{w}} \Bigg)^\top \frac{\partial L}{\partial \mathbf{z}}, d\mathbf{w} \right \rangle + 
\left \langle \frac{\partial F}{\partial L} \Bigg( \frac{\partial \mathbf{z}}{\partial \mathbf{b}} \Bigg)^\top \frac{\partial L}{\partial \mathbf{z}}, d\mathbf{b} \right \rangle \implies 
$$

$$ 
\begin{dcases}
\frac{\partial F}{\partial \mathbf{w}} = \frac{\partial F}{\partial L} \Bigg( \frac{\partial \mathbf{z}}{\partial \mathbf{w}} \Bigg)^\top \frac{\partial L}{\partial \mathbf{z}} = 1 \cdot X^\top \cdot \frac{\partial L}{\partial \mathbf{z}} \\
\\
\frac{\partial F}{\partial \mathbf{b}} = \frac{\partial F}{\partial L} \Bigg( \frac{\partial \mathbf{z}}{\partial \mathbf{b}} \Bigg)^\top \frac{\partial L}{\partial \mathbf{z}} = 1 \cdot I_n \cdot \frac{\partial L}{\partial \mathbf{z}}
\end{dcases}
$$


In [ ]:
class Per_Lin_Softmax_CE_Gradient_Function(autograd.Function):
    @staticmethod
    def __model(x, W, b):
        z = linear.linear(x, W, b)
        return softmax.softmax(z)
    
    @staticmethod
    def __loss(p, t):
        return ce.ce(p, t)
    
    @staticmethod
    def predict(X, w, b):
        with tr.no_grad():
            return Per_Lin_Softmax_CE_Gradient_Function.__model(X, w, b).argmax(dim=1)

    @staticmethod
    def forward(ctx, X, w, b, t):
        p = Per_Lin_Softmax_CE_Gradient_Function.__model(X, w, b)
        L = Per_Lin_Softmax_CE_Gradient_Function.__loss(p, t)
        
        ctx.save_for_backward(X, p, t)

        return L
    
    @staticmethod
    def backward(ctx, dF_dL):
        X, p, t = ctx.saved_tensors
        S = X.shape[0]

        dF_dL = dF_dL.to(p.dtype)
        dL_dz = (p - t) / S
        dF_dw = dF_dL * (X.t() @ dL_dz)
        dF_db = dF_dL * dL_dz.sum(dim=0)

        return (None, dF_dw, dF_db, None)

    

class Per_Lin_Softmax_CE_Gradient(nn.Module):
    """
    The version exposing the exact mechanics of gradient computation and giving 
    full control over how the model participates in PyTorch's autograd system.
    """

    class _Linear:
        """Helper class to keep the model internal layout consistent across all variants."""

        def __init__(self, w, b):
            self.weight = w
            self.bias = b

    # This is helper for testing to indicate that the `forward` method expects both, `x` and `t`.
    CUSTOM_GRAD=True

    def __init__(self, in_features: int, out_features: int, W: tr.Tensor = None, b: tr.Tensor = None) -> None:
        super().__init__()

        # `weight` has shape `(out_features, in_features)` to be `nn.Linear` compatible
        if W is None:
            self.weight = nn.Parameter(tr.randn(out_features, in_features, dtype=tr.float32)) * 0.01
        else:
            self.weight = nn.Parameter(W.clone().detach().requires_grad_(True))

        if b is None:
            self.bias = nn.Parameter(tr.randn(out_features, dtype=tr.float32))
        else:
            self.bias = nn.Parameter(b.clone().detach().requires_grad_(True))

        self.linear = Per_Lin_Softmax_CE_Gradient._Linear(self.weight, self.bias)

    def model(self, X):
        with tr.no_grad():
            return Per_Lin_Softmax_CE_Gradient_Function.__model(X, self.weight.t(), self.bias)
    
    def loss(self, p, t):
        with tr.no_grad():
            return Per_Lin_Softmax_CE_Gradient_Function.__loss(p, t)

    def predict(self, X):
        with tr.no_grad():
            return Per_Lin_Softmax_CE_Gradient_Function.predict(X, self.weight.t(), self.bias)
        
    def forward(self, X, t):
        return Per_Lin_Softmax_CE_Gradient_Function.apply(X, self.weight.t(), self.bias, t)

In [ ]:
if __name__ == "__main__":

    def onehot(z: tr.Tensor) -> tr.Tensor:
        S, C = z.shape
        p = tr.softmax(z, dim=1)
        idx = p.argmax(dim=1)
        t = tr.zeros_like(p)
        t[tr.arange(S), idx] = 1.0

        return t

    test_checkgrad_2(Per_Lin_Softmax_CE_Gradient_Function, 2, 2, 2, prec=0.01, fY=onehot)
    test_checkgrad_2(Per_Lin_Softmax_CE_Gradient_Function, 3, 3, 3, prec=0.01, fY=onehot)
    test_checkgrad_2(Per_Lin_Softmax_CE_Gradient_Function, 5, 5, 5, prec=0.01, fY=onehot)

    test_compare_2(Per_Lin_Softmax_CE_Autograd, Per_Lin_Softmax_CE_Backward, 1, 1, 1, fY=onehot)
    test_compare_2(Per_Lin_Softmax_CE_Autograd, Per_Lin_Softmax_CE_Backward, 2, 3, 4, fY=onehot)
    test_compare_2(Per_Lin_Softmax_CE_Autograd, Per_Lin_Softmax_CE_Backward, 10, 20, 30, fY=onehot)

    test_compare_2(Per_Lin_Softmax_CE_Backward, Per_Lin_Softmax_CE_Gradient, 1, 1, 1, fY=onehot)
    test_compare_2(Per_Lin_Softmax_CE_Backward, Per_Lin_Softmax_CE_Gradient, 2, 3, 4, fY=onehot)
    test_compare_2(Per_Lin_Softmax_CE_Backward, Per_Lin_Softmax_CE_Gradient, 10, 20, 30, fY=onehot)